# Plotting Parrots

## Initialising codebase

In [23]:
import sys
import os
from datetime import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import plotly.express as px

### Import data

In [24]:

beaufort_scale = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/beaufort_scale.csv')
birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv', low_memory=False)
sea_states = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/sea_states.csv')
ships = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/ships.csv')

### Check if data imported correctly

### Beaufort_scale

In [25]:
beaufort_scale.head()

,wind_speed_class,wind_description,wind_speed_knots_min,wind_speed_knots_max
0,0,calm,0,1.0
1,1,light air,1,3.0
2,2,light breeze,4,6.0
3,3,gentle breeze,7,10.0
4,4,moderate breeze,11,16.0


### Birds

In [26]:
birds.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,sitting_on_ship,in_hand,n_flying_past,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,False,False,0.0,False,6.0,True,0.0,False,NaN,False
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,False,False,0.0,False,8.0,True,0.0,False,NaN,False
3,4,1083001,Fairy prion,Pachyptila turtur,PACTUR,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,False,False,0.0,False,4.0,True,0.0,False,NaN,False


In [27]:
sea_states.head()

,sea_state_class,sea_state_description,wave_meters_min,wave_meters_max
0,0,"calm, glassy",0.0,0.0
1,1,"calm, rippled",0.0,0.1
2,2,smooth,0.1,0.5
3,3,slight,0.5,1.2
4,4,moderate,1.3,2.5


In [28]:
ships.head()

,record_id,date,time,latitude,longitude,hemisphere,activity,speed,direction,cloud_cover,...,wind_speed_class,wind_direction,air_temperature,pressure,sea_state_class,sea_surface_temperature,depth,observer,census_method,season
0,1083001,1975-10-15,14:00:00,-45.917,165.400,E,"steaming, sailing",15.0,NaN,overcast,...,5.0,NaN,NaN,NaN,5.0,NaN,NaN,D. Jeffcock,full,spring
1,1084001,1975-11-03,13:10:00,-35.533,125.000,E,"steaming, sailing",14.0,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
2,1084002,1975-11-04,14:20:00,-37.667,132.250,E,"steaming, sailing",14.5,NaN,overcast,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
3,1084003,1975-11-08,16:15:00,-40.000,162.000,E,"steaming, sailing",14.6,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
4,1086001,1975-11-16,12:30:00,-36.167,174.917,E,"steaming, sailing",15.0,NaN,partially cloudy,...,0.0,NaN,NaN,NaN,1.0,NaN,NaN,D. Jeffcock,full,spring


The data was recorded sing guidelines for the Australian Seabird Mapping Scheme and counts seabirds observed from a ship for a duration of time, with a full observations being 10 minutes.

The analysis uses 4 datasets: 
1. birds.csv, containing individual seabird observation records; 
2. ships.csv, containing information about ship-based censuses, including geographical coordinates, dates and seasons, and various weather conditions like sea state and wind speed; 
3. beaufort_scale.csv, containing description of the wind speed corresponsing to the integers in the beaufort scale class in ships.csv; 
4. sea_states.csv, containing description of the sea states corresponding to the integers in the sea state class in ships.csv.

### Question of interest: How is the observed distribution of seabirds associated with geographical location, season and weather conditions?
The study investigates the question: **How is the observed distribution of seabirds associated with geographical location, season and weather conditions?** These factors may influence seabird occurrence through differences in habitat, migration patterns and environmental conditions. To investigate these relationships, the analysis focuses on the five most frequently encountered seabird genera, allowing meaningful comparisons while maintaining clear visualisations. Three complementary visualisations are used to examine geographical, seasonal and weather-related patterns in seabird encounters.

# Data Cleaning

To ensure consistency across the analysis, seabird observations were classified by genus using scientific names standardised to the IOC World Bird List v15.2. This included correcting inconsistent names and reassigning species whose genus classifications had changed. Records with missing scientific names or ambiguous identifications spanning multiple genera were excluded to avoid incorrect classification.

Only full ship censuses were retained, as partial censuses may involve shorter or less systematic observation periods and are therefore less comparable. This reduced the dataset from 12,310 to 12,111 censuses. Bird observations were then matched to the retained censuses using record_id.

Observations with missing bird counts were retained for encounter-rate calculations because the presence of a bird was still recorded, even when its abundance was unknown. Similarly, counts of 99,999 were retained as they represent very large observed flocks rather than missing values. However, these capped counts limit the accuracy of abundance-based statistics, supporting the use of encounter rate as the primary measure of seabird occurrence.

## Genus names
Genus is the first word of `species_scientific_name`, standardised to the [IOC World Bird List v15.2](https://www.worldbirdnames.org/) (fixes misspellings, renamed genera and species that moved genus).

* Rows with no scientific name are unidentified birds, so they are dropped.
* Names that span two genera (e.g. "Procellaria parkinsoni / Puffinus carneipes") are dropped.
* "Puffinus sp." (Shearwater, unidentified) and "Sterna sp." (Tern, unidentified) are dropped: after the IOC split, an unidentified shearwater could be *Ardenna*, *Puffinus* or *Calonectris*, and an unidentified tern could be one of several genera.
* Rows with no `count` are **kept**: the bird was seen, only the number was not written down. They still count as a sighting, and sums or medians of `count` skip them.
* A `count` of 99999 means "estimated over 100,000" in the dataset's data dictionary, so these are real, very large flocks and are kept.

Individual seabird species were classified into their respective genera to facilitate broader comparisons across the 3 factors and reduce the complexity of the dataset.

In [29]:
birds_cleaned = birds.copy()

birds_cleaned["Genus"] = birds_cleaned["species_scientific_name"].str.split().str[0]

In [30]:
# Misspelt genus
GENUS_TYPOS = {
    "Thallassarche": "Thalassarche",
    "Phaeton":       "Phaethon",
    "Magadyptes":    "Megadyptes",
}

# Genus renamed in the IOC World Bird List
IOC_GENUS_RENAMES = {
    "Catharacta":   "Stercorarius",
    "Stictocarbo":  "Phalacrocorax",
    "Procelsterna": "Anous",
    "Lugensa":      "Aphrodroma",
}

# Changed genera by IOC
IOC_SPECIES_MOVES = {
    "Diomedea impavida":     "Thalassarche",
    "Diomedea melanophrys":  "Thalassarche",
    "Puffinus griseus":      "Ardenna",
    "Puffinus tenuirostris": "Ardenna",
    "Puffinus carneipes":    "Ardenna",
    "Puffinus pacificus":    "Ardenna",
    "Puffinus bulleri":      "Ardenna",
    "Larus novaehollandiae": "Chroicocephalus",
    "Larus bulleri":         "Chroicocephalus",
    "Sterna bergii":         "Thalasseus",
    "Sterna albifrons":      "Sternula",
    "Sterna nereis":         "Sternula",
}

def ioc_genera(name): 
    genera, genus = set(), ""
    for part in name.split("/"):
        words = part.split()
        if not words:
            continue
        # epithet is the word after genus which is required for changing the genus
        if words[0][0].isupper():   # part names its own genus: "Genus species"
            genus, epithet = words[0], (words[1] if len(words) > 1 else "") # epithet is the name after the genus
        else:                       # "Genus a / b": b shares the previous genus
            epithet = words[0]
        fixed = GENUS_TYPOS.get(genus, IOC_GENUS_RENAMES.get(genus, genus))
        genera.add(IOC_SPECIES_MOVES.get(f"{genus} {epithet}", fixed))
    return sorted(genera)   # unique genera, as a list

n_start = len(birds_cleaned)
print(f"Starting rows: {n_start} | Genera: {birds_cleaned['Genus'].nunique()}\n")

# Drop unidentified bird species
before = len(birds_cleaned)
birds_cleaned = birds_cleaned.dropna(subset=["species_scientific_name"]).copy()
print(f"1. No scientific name dropped: {before - len(birds_cleaned)} rows ({len(birds_cleaned)} remaining)\n")

# Drop names that span two genera
genera = birds_cleaned["species_scientific_name"].apply(ioc_genera)
multiple_genera = genera.apply(len) > 1
print(f"2. Spanning two genera dropped: {multiple_genera.sum()} rows ({(~multiple_genera).sum()} remaining)")
print(birds_cleaned.loc[multiple_genera, "species_scientific_name"].value_counts().to_string(), "\n")
birds_cleaned = birds_cleaned[~multiple_genera].copy()

# Assign IOC genus (typos and renames applied)
birds_cleaned["Genus_original"] = birds_cleaned["Genus"]
birds_cleaned["Genus"] = genera[~multiple_genera].str[0]
changed = birds_cleaned[birds_cleaned["Genus"] != birds_cleaned["Genus_original"]]
print(f"3. Genus changed to IOC: {len(changed)} rows | Genera: {birds_cleaned['Genus_original'].nunique()} -> {birds_cleaned['Genus'].nunique()}")
print(changed.groupby(["Genus_original", "Genus"]).size().rename("rows").to_string(), "\n")

# Drop unidentified shearwaters and terns: after the IOC split their genus is unknown
unknown_genus = birds_cleaned["species_scientific_name"].isin(["Puffinus sp.", "Sterna sp."])
print(f"4. Genus unknown after IOC split dropped: {unknown_genus.sum()} rows ({(~unknown_genus).sum()} remaining)")
print(birds_cleaned.loc[unknown_genus, "species_common_name"].value_counts().to_string(), "\n")
birds_cleaned = birds_cleaned[~unknown_genus].copy()

# Keep records without a count: the bird was seen but its number was not written down,
# so it is still a sighting. Sums and medians of "count" skip these rows automatically
print(f"5. No count kept as sightings: {birds_cleaned['count'].isna().sum()} rows\n")

# Note that count = 99999 means "estimated over 100,000" (data dictionary),
# so these rows are real, very large flocks and are kept

print(f"Total dropped: {n_start - len(birds_cleaned)} | Rows remaining: {len(birds_cleaned)} | Genera: {birds_cleaned['Genus'].nunique()}")

Starting rows: 49019 | Genera: 47

1. No scientific name dropped: 1101 rows (47918 remaining)

2. Spanning two genera dropped: 4 rows (47914 remaining)
species_scientific_name
Procellaria parkinsoni / Puffinus carneipes              2
Procellaria parkinsoni / Pterodroma macroptera gouldi    1
Eudyptes / Magadyptes sp                                 1 

3. Genus changed to IOC: 10611 rows | Genera: 47 -> 46
Genus_original  Genus          
Catharacta      Stercorarius        168
Diomedea        Thalassarche       3610
Larus           Chroicocephalus     788
Lugensa         Aphrodroma           10
Phaeton         Phaethon              5
Procelsterna    Anous                27
Puffinus        Ardenna            5135
Sterna          Sternula              3
                Thalasseus          402
Stictocarbo     Phalacrocorax        33
Thallassarche   Thalassarche        430 

4. Genus unknown after IOC split dropped: 130 rows (47784 remaining)
species_common_name
Shearwater (unidentified)  

## Ships
Keep full counts only. Partial counts are shorter or casual observations, so they are not comparable. We drop all records from ships.csv with census column "partial"

In [31]:
ships_cleaned = ships.copy()
print(f"Starting counts: {len(ships_cleaned)}\n")

# Keep full counts only 
partial = ships_cleaned["census_method"] != "full"
print(f"1. Partial counts dropped: {partial.sum()} ({(~partial).sum()} remaining)")
ships_cleaned = ships_cleaned[~partial].copy()

Starting counts: 12310

1. Partial counts dropped: 199 (12111 remaining)


## Match birds to the cleaned ships
Keep only bird records from the counts kept above.

In [32]:
# Keep birds from full counts only
before = len(birds_cleaned)
birds_cleaned = birds_cleaned[birds_cleaned["record_id"].isin(ships_cleaned["record_id"])].copy()
print(f"Bird rows from dropped counts removed: {before - len(birds_cleaned)} ({len(birds_cleaned)} remaining)")

Bird rows from dropped counts removed: 536 (47248 remaining)


In [ ]:
# Count each unique genus per census (min_count=1 keeps the total as NaN when no count was written down)
genus_by_count = birds_cleaned.groupby(["record_id", "Genus"], as_index=False)["count"].sum(min_count=1)
genus_by_count.head()

# Top 5 genera
Every plot uses the same five genera, picked here from the cleaned data. Use `top5_genus` for the list and `top5_birds` for their bird records.

## How the top 5 are picked
The top 5 genera were chosen by encounter rate, which is the share of all full censuses in which the genus was seen at least once. Ranking by it puts a genus seen often above one seen in a few huge flocks. This avoids giving disproportionate weight to exceptionally large flocks, particularly because the dataset records flocks estimated at over 100,000 birds as 99999.

In [33]:
# Count how many census records there are for each genus
birds_cleaned["n_census"] = (birds_cleaned.groupby("Genus")["record_id"]
                             .transform("nunique"))

In [34]:
# Summarise each genus: how many censuses it was seen in, total birds, and typical flock size
genus_summary = (genus_by_count.groupby("Genus")
                 .agg(n_census=("record_id", "nunique"),
                      total_count=("count", "sum"),
                      median_flock=("count", "median"))
                 .reset_index())

genus_summary["count_per_census"] = genus_summary["total_count"] / genus_summary["n_census"]

# Encounter rate: divide by every full census (including censuses with no birds), not only censuses that logged a bird
total_census = ships_cleaned["record_id"].nunique()
genus_summary["encounter_rate"] = genus_summary["n_census"] / total_census * 100

top5_genus = genus_summary.nlargest(5, "encounter_rate")
top5_genus[["Genus", "n_census", "encounter_rate"]]

,Genus,n_census,encounter_rate
9,Diomedea,5806,47.939889
42,Thalassarche,4540,37.486582
3,Ardenna,4074,33.638841
35,Pterodroma,3987,32.920486
8,Daption,2945,24.316737


In [35]:
# Bird records of the top 5 genera only (from full counts)
top5_birds = birds_cleaned[birds_cleaned["Genus"].isin(top5_genus["Genus"])].copy()
top5_birds.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding,Genus,Genus_original,n_census
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,False,6.0,True,0.0,False,NaN,False,Diomedea,Diomedea,5806
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,False,2.0,True,0.0,False,NaN,False,Thalassarche,Diomedea,4540
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,False,8.0,True,0.0,False,NaN,False,Daption,Daption,2945
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,False,4.0,True,0.0,False,NaN,False,Ardenna,Puffinus,4074
5,6,1084001,Royal albatross sensu lato,Diomedea epomophora / sanfordi,DIOEPOSAN,NaN,NaN,NaN,NaN,10.0,...,False,0.0,False,8.0,True,NaN,False,Diomedea,Diomedea,5806


### Summary statistics
Following the removal of partial censuses, 12,111 full censuses were retained for analysis. The five genera with the highest encounter rates were selected, with Diomedea being the most frequently encountered (47.9% of censuses), followed by Thalassarche (37.5%), Ardenna (33.6%), Pterodroma (32.9%) and Daption (24.3%).

Notably, Ardenna recorded approximately 1.24 million birds despite ranking only third in encounter frequency, illustrating how occasional large aggregations can substantially influence total abundance. This supports the use of encounter rate to select frequently observed genera without allowing exceptionally large flocks to dominate the ranking.

# Plotting by geographical location

## Purpose
This visualisation investigates how seabird encounter frequency varies across geographical locations and whether different genera exhibit distinct spatial distribution patterns. 

### Variables used
Latitude and longitude from ships.csv determine the geographical position of each census, while record_id and Genus identify whether a particular genus was encountered during that census. Calendar quarter is also incorporated into the calculation to account for differences in seasonal survey coverage.

### Reason for geographical heatmap
An interactive geographical heatmap was selected because it allows spatial variations in seabird encounter frequency to be examined directly in their geographical context. Colour intensity represents the percentage of censuses recording each genus, with darker red indicating higher encounter rates. This makes it easier to identify geographical hotspots and compare coastal and offshore distribution patterns.

Unlike a conventional scatter plot, which may suffer from overlapping observations along frequently travelled shipping routes, the heatmap summarises nearby observations into a continuous spatial representation. The interactive dropdown allows users to switch between genera, while a consistent colour scale ensures that encounter rates remain comparable.

In [36]:
birds_by_geo_loc = birds_cleaned.merge(ships_cleaned[["record_id", "latitude", "longitude"]], on="record_id", how="left")
top5_birds_by_geo_loc = birds_by_geo_loc[birds_by_geo_loc["Genus"].isin(top5_genus["Genus"])]
top5_birds_by_geo_loc.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,accompanying,n_following_ship,following_ship,moulting,naturally_feeding,Genus,Genus_original,n_census,latitude,longitude
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,True,0.0,False,NaN,False,Diomedea,Diomedea,5806,-45.917,165.4
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,True,0.0,False,NaN,False,Thalassarche,Diomedea,4540,-45.917,165.4
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,True,0.0,False,NaN,False,Daption,Daption,2945,-45.917,165.4
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,True,0.0,False,NaN,False,Ardenna,Puffinus,4074,-45.917,165.4
5,6,1084001,Royal albatross sensu lato,Diomedea epomophora / sanfordi,DIOEPOSAN,NaN,NaN,NaN,NaN,10.0,...,False,8.0,True,NaN,False,Diomedea,Diomedea,5806,-35.533,125.0


## Data transformation
To reduce the influence of unequal survey effort, encounter rates were calculated using the proportion of censuses recording each genus rather than raw sighting counts. Census observations were grouped into 0.1° geographical cells, and Gaussian smoothing was applied to estimate encounter rates using nearby observations. Rates were calculated separately for each calendar quarter before being averaged equally, preventing more heavily sampled quarters from disproportionately influencing the geographical patterns.

Only regions with at least six nearby censuses in every quarter were displayed. Consequently, blank areas indicate insufficient sampling coverage rather than the absence of seabirds.

### Step 1: one row per count, with its location and quarter
* Keep only **full** counts (12,111 of 12,310). The 199 partial counts are shorter, so they are not comparable.
* Drop the 11 full counts with no coordinates, leaving **12,100**.
* Tag each count with its quarter of the year: Jan–Mar (3,457 counts), Apr–Jun (2,123), Jul–Sep (2,506), Oct–Dec (4,014).
* For each count and genus, record True if the genus was logged at least once. A genus counts once per count, however many birds were seen.

In [37]:
# full censuses with coordinates, tagged with their quarter of the year
geo_loc = ships_cleaned[["record_id", "latitude", "longitude", "date"]].dropna()
geo_loc["quarter"] = pd.to_datetime(geo_loc["date"]).dt.quarter          # 1 = Jan–Mar ... 4 = Oct–Dec

In [38]:
# one row per census, True/False for each group
present = top5_birds_by_geo_loc.assign(present=True).pivot_table(index="record_id", columns="Genus", values="present",
                                                         aggfunc="any", fill_value=False)
cells = geo_loc.merge(present, left_on="record_id", right_index=True, how="left").fillna(False)

### Step 2: how often each genus was seen, one map at a time
**Measure: presence rate**, the share of 10-minute counts in a patch of sea that recorded the genus. We use a rate rather than raw sightings because ships surveyed some lanes far more often than others, so raw sightings would mostly map ship traffic. We chose presence over average birds per count because flock sizes are extremely skewed (Ardenna: mean 312 birds per sighting, median 4), so a few giant flocks would decide the colour of a whole region.

1. **Grid.** Counts are binned into 0.1° cells (about 10 km).
2. **Smooth.** Both grids (all counts, and counts with the genus) are blurred with a Gaussian filter (σ = 6 cells ≈ 0.6°), and the rate is one divided by the other. Each patch therefore draws on counts within roughly 60 km, which fills the gaps between neighbouring ship tracks without inventing data far from any ship.
3. **Weight the seasons equally.** The rate is worked out separately for each quarter, then the four are averaged. Without this, the Tasman Sea (39% of its counts in Jan–Mar but only 14% in Jul–Sep) would look like a summer place, and season would leak into a map that is meant to be about location.
4. **Only colour well-surveyed sea.** A cell is shown only if it had at least 6 counts nearby in **every** quarter. Blank sea means "not enough data", and not "no birds". About 27% of the map window fulfills the criteria, and the coloured areas follow the ship routes.
5. **Map window.** 147–180°E, 30–48.5°S keeps 86% of all full counts, including the Tasman crossings between Australia and New Zealand.


In [39]:
from scipy.ndimage import gaussian_filter
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from shapely.geometry import box
np.seterr(divide="ignore", invalid="ignore")

LON, LAT = (147, 180), (-48.5, -30) # map window zoomed in to cover only NZ surroundings and tasman sea between NZ and AU
lon_bins, lat_bins = np.arange(145, 182, 0.1), np.arange(-50.5, -28, 0.1)   # 0.1° grid, 2° spare around the window

def smooth_count(df, weights=None):
    """Count censuses in each 0.1° cell, then blur over ~0.6° (6 cells) so gaps between ship tracks fill in."""
    counts, _, _ = np.histogram2d(df["latitude"], df["longitude"], bins=[lat_bins, lon_bins], weights=weights)
    return gaussian_filter(counts, sigma=6) * 226

quarters = [cells[cells["quarter"] == q] for q in [1, 2, 3, 4]]
effort = [smooth_count(df) for df in quarters]
surveyed = np.all([e >= 6 for e in effort], axis=0)  # only keep sea with >= 6 censuses nearby in every quarter

rate = {}
for g in top5_genus["Genus"]:
    # % of censuses that logged g, per quarter, then the average of the four quarters (each season has equal weightage)
    per_quarter = [smooth_count(df, df[g].astype(float)) / e * 100 for df, e in zip(quarters, effort)]
    rate[g] = np.where(surveyed, np.mean(per_quarter, axis=0), np.nan)

lon_mid, lat_mid = lon_bins[:-1] + 0.05, lat_bins[:-1] + 0.05
rows = np.where((lat_mid >= LAT[0]) & (lat_mid <= LAT[1]))[0][::2]
cols = np.where((lon_mid >= LON[0]) & (lon_mid <= LON[1]))[0][::2]
inside = np.ix_(rows, cols)
nearby = sum(effort)[inside].round().astype("float32")
order = ["Diomedea", "Thalassarche", "Ardenna", "Pterodroma", "Daption"]


In [40]:
# trace coastline (one outline trace, simplified so the HTML stays small)
land = gpd.read_file("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_land.geojson")
land = land.clip(box(LON[0] - 3, LAT[0] - 3, LON[1] + 3, LAT[1] + 3)).explode().simplify(0.02)
coast_x, coast_y = [], []
for poly in land:
    x, y = poly.exterior.xy
    coast_x += list(x) + [None]; coast_y += list(y) + [None]

# distance from each grid cell's centre to the nearest coastline (used for the red observation line and Step 4)
coast = land.union_all()
grid = gpd.GeoSeries(gpd.points_from_xy(*[a.ravel() for a in np.meshgrid(lon_mid, lat_mid)]))
km = grid.distance(coast).to_numpy().reshape(surveyed.shape) * 100   # 1° is about 100 km at these latitudes
in_window = np.outer((lat_mid >= LAT[0]) & (lat_mid <= LAT[1]), (lon_mid >= LON[0]) & (lon_mid <= LON[1]))
mapped = surveyed & in_window               # the coloured 0.1° cells on the map
near = mapped & (km > 0) & (km <= 50)       # coloured cells within 50 km of land (cells centred on land left out)
far = mapped & (km > 150)                   # coloured cells more than 150 km from land

## Visualisation

In [41]:
names = {"Diomedea": "great albatrosses", "Thalassarche": "mollymawks", "Ardenna": "large shearwaters",
         "Pterodroma": "gadfly petrels", "Daption": "Cape petrel"}
n_near = {g: np.nanmedian(rate[g][near]) for g in order}    # median % over coloured cells within 50 km of land
n_far = {g: np.nanmedian(rate[g][far]) for g in order}      # median % over coloured cells more than 150 km from land

# title (main pattern) and red line (hotspot) for each view, written from the maps and the Step 4 table
headline = {"Diomedea": "Great albatrosses (<i>Diomedea</i>) are seen most often far out at sea",
            "Thalassarche": "Mollymawks (<i>Thalassarche</i>) are seen as often near the coast as far out at sea",
            "Ardenna": "Large shearwaters (<i>Ardenna</i>) are seen most often near the coast",
            "Pterodroma": "Gadfly petrels (<i>Pterodroma</i>) are seen more often far out at sea",
            "Daption": "Cape petrels (<i>Daption</i>) gather off the South Island's east coast",
            "All": "Albatrosses and gadfly petrels favour the open sea; large shearwaters stay near the coast"}
hotspot = {"Diomedea": "Highest across the open Tasman Sea.",
           "Thalassarche": "Slightly higher off the south-east of the South Island.",
           "Ardenna": "Hotspots off north-east North Island, southern NZ and the Australian coast.",
           "Pterodroma": "One coastal exception: a hotspot off north-east North Island.",
           "Daption": "Strongest off the east coast of the South Island.",
           "All": "Mollymawks show no coast-to-sea difference; Cape petrels are concentrated off the South Island's east coast."}
def title(view):
    what = "each genus" if view == "All" else "the genus"
    return (f"<b>{headline[view]}</b><br><span style='font-size:13px;color:dimgrey'>Share of 10-minute counts from ships that recorded "
            f"{what}, smoothed over about 60 km · full counts 1969–1990 · each season weighted equally</span>"
            f"<br><span style='font-size:13px;color:#7f0000'><b>{hotspot[view]}</b></span>")

# axis 1 = one big map, axes 2-6 = the five small maps for "All genera"; each trace is tagged (meta) with the view it belongs to
fig = go.Figure()
counts_nearby = nearby.round().astype(int).tolist()    # plain list, so every plotly version can show it on hover
def add_map(g, axis, tag, bar):
    a = "" if axis == 1 else axis                      # plotly names the axes x, x2, x3 ...
    fig.add_trace(go.Contour(x=lon_mid[cols], y=lat_mid[rows], z=rate[g][inside].round().astype("float32"), customdata=counts_nearby,
                             zmin=0, zmax=100, contours=dict(start=0, end=100, size=12.5), autocontour=False, line_width=0,
                             colorscale="OrRd", hoverongaps=False, xaxis=f"x{a}", yaxis=f"y{a}", meta=tag, showscale=bar is not None,
                             colorbar=dict(**(bar or {}), orientation="h", thickness=14, dtick=20, ticksuffix="%",
                                           title=dict(text="% of counts that recorded the genus", side="top")),
                             hovertemplate=f"<b><i>{g}</i></b> ({names[g]})<br>Recorded in %{{z:.0f}}% of counts here<br>"
                                           "Based on %{customdata} counts within about 60 km<extra></extra>"))
def add_land(axis, tag):
    a = "" if axis == 1 else axis
    fig.add_trace(go.Scatter(x=coast_x, y=coast_y, fill="toself", fillcolor="white", mode="lines", line=dict(color="#5b6470", width=0.6),
                             hoverinfo="skip", showlegend=False, xaxis=f"x{a}", yaxis=f"y{a}", meta=tag))

for g in order:                                        # big map: one colour layer per genus, colour scale underneath
    add_map(g, 1, g, dict(x=0.37, xanchor="center", y=0, yanchor="bottom", len=0.4))
add_land(1, "single")
for k, g in enumerate(order):                          # small maps: one colour scale only, in the empty 6th space
    add_map(g, k + 2, "All", dict(x=0.84, xanchor="center", y=0.03, yanchor="bottom", len=0.28) if k == 0 else None)
    add_land(k + 2, "All")

def visible(view):
    return [t.meta == view or (t.meta == "single" and view != "All") for t in fig.data]

# boxes and labels
small = [([x, x + 0.32], [y, y + 0.44]) for y in (0.53, 0.03) for x in (0, 0.34, 0.68)][:5]   # where the small maps go
def note(x, y, text, box=True, **kw):  # text on the figure; box=True gives it a white box with a grey border
    a = dict(x=x, y=y, xref="paper", yref="paper", xanchor="left", yanchor="top", align="left", showarrow=False, text=text,
             font=dict(size=12, color="#333"))
    if box:
        a.update(bgcolor="white", bordercolor="#d0d6dc", borderwidth=1, borderpad=6)
    return a | kw
read_single = note(0.775, 0.97, "<b>How to read</b><br>Colour = % of 10-minute counts in<br>that patch of sea that recorded the<br>"
                   "genus. Darker red = recorded more often.<br><br>Blank sea = too few counts in at least<br>one season, not \"no birds\".<br>"
                   "Coloured areas follow the ship routes.<br><br>Corner box = median % across all<br>coloured cells within 50 km of the<br>"
                   "coast, and across those over 150 km<br>away (full table in Step 4).<br><br>Hover over the map for the exact %<br>"
                   "and the number of counts behind it.<br><br>Use the menu above to switch genus<br>or compare all five.")
read_all = note(0.69, 0.45, "<b>How to read</b><br>Colour = % of 10-minute counts in that<br>patch of sea that recorded the genus.<br>"
                "Darker red = recorded more often.<br>All five maps share one colour scale.<br>Blank sea = too few counts in a season.<br>"
                "Under each map: median % across coloured<br>cells within 50 km / over 150 km of land.<br><br>"
                "Hover for exact numbers, or pick one<br>genus in the menu to see it larger.")
def places(axis):   # place names on map axis 1 (big map), or short names on axes 2-6 (small maps)
    a, big = ("", True) if axis == 1 else (axis, False)
    return [dict(x=x, y=y, xref=f"x{a}", yref=f"y{a}", text=t, showarrow=False, font=dict(size=12 if big else 9, color="#6b7280"))
            for x, y, t in [(150, -31.3, "AUSTRALIA" if big else "AUS"), (176.3, -38.4, "NEW<br>ZEALAND" if big else "NZ"),
                            (157, -42.7, "<i>Tasman Sea</i>")]]
views = {g: places(1) + [read_single, note(0.01, 0.115, f"<b>Median % of counts that recorded the genus</b><br><span style='color:dimgrey'>across all "
                                f"coloured cells in each zone</span><br>Within 50 km of the coast: <b>{n_near[g]:.0f}%</b><br>"
                                f"More than 150 km from the coast: <b>{n_far[g]:.0f}%</b>", yanchor="bottom")] for g in order}
views["All"] = [read_all] + [p for k in range(5) for p in places(k + 2)] + [a for k, g in enumerate(order) for a in (
    note(small[k][0][0], small[k][1][1], f"<b><i>{g}</i></b> <span style='color:dimgrey'>({names[g]})</span>", box=False, yanchor="bottom"),
    note(small[k][0][0], small[k][1][0] + 0.008, f"Median %: within 50 km of coast <b>{n_near[g]:.0f}%</b> · over 150 km "
         f"<b>{n_far[g]:.0f}%</b>", box=False, yanchor="bottom", font=dict(size=11, color="#333")))]

# axes not in use are shrunk into a tiny corner, if not their invisible area sits on top of the map and blocks hover
def domains(view):
    big, tiny = ([0, 0.76], [0.1, 1]), ([0.999, 1], [0, 0.001]) # big map: left side, bottom 10% kept for its colour scale
    out = {}
    for i, (xd, yd) in enumerate([tiny if view == "All" else big] + [s if view == "All" else tiny for s in small], start=1):
        a = "" if i == 1 else i
        out[f"xaxis{a}"] = dict(domain=xd, range=LON, visible=False, constrain="domain", constraintoward="left")
        out[f"yaxis{a}"] = dict(domain=yd, range=LAT, visible=False, constrain="domain", constraintoward="top", scaleanchor=f"x{a}", scaleratio=1.3)
    return out

buttons = [dict(label="All genera (compare)" if v == "All" else f"{v} ({names[v]})", method="update",
                args=[{"visible": visible(v)}, {"title.text": title(v), "annotations": views[v],
                                                **{f"{k}.domain": d["domain"] for k, d in domains(v).items()}}]) for v in views]
for t, on in zip(fig.data, visible(order[0])):         # start on the first genus
    t.visible = on
fig.update_layout(domains(order[0]), annotations=views[order[0]], width=1050, height=770, margin=dict(l=10, r=10, t=140, b=10),
                  title=dict(text=title(order[0]), x=0.01, y=0.975, yanchor="top", font=dict(size=18)),
                  updatemenus=[dict(buttons=buttons, x=0, xanchor="left", y=1.01, yanchor="bottom", bgcolor="white", font=dict(size=13))],
                  plot_bgcolor="#e9eef2", paper_bgcolor="white")
fig.show()


**Reading the plot**
* **One genus at a time.** Use the dropdown to pick a genus, so each map is large and easy to read. The last option, **All genera (compare)**, shows the five side by side on the same colour scale.
* **Title** gives the main pattern for the map on screen. The grey line under it says what the colour measures.
* **Red line** says where the genus's main hotspot is.
* **Coast vs open-sea numbers** (corner box on the single map, a line under each compare map) give the median rate across all coloured 0.1° cells on the map whose centre is within 50 km of the coastline, and across those more than 150 km from it (distances from the cell below). The full table is in Step 4.
* **How to read box** (right of the single map, or in the empty sixth space of the compare view) explains the colours and blank sea. The colour scale sits under the single map, and under the box in the compare view.
* **Hover** over a coloured patch to see its rate and how many counts it is based on.

### Step 4: near the coast vs open sea
To put numbers on the pattern in the map, each coloured 0.1° grid cell on the map is given its distance to the nearest coastline and placed in a band: < 50 km, 50–150 km or > 150 km. The table gives each genus's median rate in each band. Distances are approximate (1° ≈ 100 km at these latitudes); cells whose centre falls on land are left out.

In [42]:
# Same rates, split by distance from the coast (km was worked out in Step 3)
band = pd.cut(km[mapped], [0, 50, 150, np.inf], labels=["< 50 km", "50–150 km", "> 150 km"])
pd.DataFrame({g: rate[g][mapped] for g in order}).groupby(band, observed=True).median().round(0).T

,< 50 km,50–150 km,> 150 km
Diomedea,23.0,40.0,80.0
Thalassarche,37.0,31.0,37.0
Ardenna,42.0,40.0,18.0
Pterodroma,21.0,28.0,44.0
Daption,19.0,19.0,25.0


### Discussion
The geographical heatmap reveals distinct spatial patterns in seabird encounter frequency, suggesting that geographical location is associated with the occurrence of different genera. *Diomedea* exhibits a particularly strong offshore pattern, with a median encounter rate of 80% in mapped areas more than 150 km from the coast, compared with only 23% within 50 km of the coast. In contrast, *Ardenna* shows the opposite trend, with a higher median encounter rate near the coast (42%) than offshore (18%). *Pterodroma* also exhibits higher offshore encounter rates (44%) than coastal rates (21%), whereas *Thalassarche* shows relatively little difference between these regions (37% in both).

These contrasting patterns may reflect differences in foraging strategies, habitat use and prey availability among seabird genera. For example, the higher offshore encounter rates of *Diomedea* may be consistent with their ability to travel long distances over open waters, while the greater coastal occurrence of *Ardenna* may reflect the availability of food resources in particular coastal marine environments. However, these explanations remain hypotheses, as the dataset does not directly measure prey availability or habitat quality.

Although encounter rates account for differences in census frequency, the observed patterns may still be influenced by shipping routes, detection conditions and differences in survey coverage across years. Therefore, the results demonstrate geographical associations in recorded seabird occurrence rather than establishing that geographical location directly causes differences in seabird distribution.

### Checks and limitations
* **Sensitivity.** We re-ran the map with a narrower and a wider blur (σ = 4 or 8 cells) and a lower and a higher effort threshold (3 or 12 counts per quarter). The near-coast vs open-sea contrast barely moves: Diomedea stays at 22–27% near the coast vs 80–81% in the open sea, Ardenna 42–45% vs 17–18%, and Pterodroma 20–22% vs 44–50%; Thalassarche stays flat and Daption slightly higher offshore. The median rate over all coloured sea moves more (Diomedea 43–64%), because these settings change how much near-shore versus open sea is coloured.
* **Data collection is concentrated around common shipping routes, and is not a random sample.** Ships followed fixed lanes, so the map describes the sea along those routes, not the whole region.
* **Years differ by area.** 76% of Tasman counts are from 1975–1984, while 89% of counts in New Zealand waters are from 1980–1989, so part of a difference between areas could be a difference between decades, ships or observers.
* **Rounded positions.** About 43% of counts give their position only to the nearest whole degree (about 100 km), which is why a few counts fall on land. The 0.6° blur and the 6-counts-per-quarter rule soften this, but patterns smaller than about 100 km should not be over-read.
* **Data comes from observations, and does not depict true abundance.** Large birds and birds that follow ships are easier to spot than small ones, so rates compare how often a genus was *recorded*. A rate also ignores flock size: 2 birds and 2,000 birds count the same.
* **Location is not independent from weather.** Wind and sea state vary from place to place, which overlaps slightly with the weather plot.

# Seasonal analysis — two chart options

This section continues directly from the shared cleaning and top-five genus selection above. It uses `genus_by_count`, `ships_cleaned`, and `top5_genus`; it does **not** re-run taxonomy cleaning. Both charts show the same seasonal encounter-rate percentages, just in different formats.

**Definition:** percentage of full censuses in each season recording a genus at least once. The denominator includes full censuses with no record of that genus. A sighting with an unknown bird count still counts as an encounter.


## 1. Connect genus sightings to seasons
`genus_by_count` has one row per genus per survey; `ships_cleaned` has the season for each full census. Merge by `record_id` to attach the season. The many-to-one check protects against duplicate ship IDs.

In [ ]:
season_birds = genus_by_count.merge(
    ships_cleaned[["record_id", "season"]],
    on="record_id", how="inner", validate="many_to_one"
)
selected_genera = top5_genus["Genus"].tolist()
season_birds = season_birds[season_birds["Genus"].isin(selected_genera)].copy()
print("Selected genera:", selected_genera)

## 2. Count encounters and eligible surveys
The numerator is the number of distinct full censuses recording a genus in a season. The denominator is all full censuses in that season, including those without the genus. This controls for unequal survey counts across seasons.

In [ ]:
season_order = ["spring", "summer", "autumn", "winter"]
encounters = (season_birds.groupby(["season", "Genus"])["record_id"]
              .nunique().rename("n_encounters"))
season_n = (ships_cleaned.groupby("season")["record_id"]
            .nunique().reindex(season_order).rename("total_censuses"))
print(season_n)

## 3. Calculate the percentages
Create every season × genus combination, filling genuinely absent encounters with zero. Divide by the number of full censuses in the corresponding season.

In [ ]:
pairs = pd.MultiIndex.from_product(
    [season_order, selected_genera], names=["season", "Genus"]
)
season_summary = encounters.reindex(pairs, fill_value=0).reset_index()
season_summary = season_summary.merge(
    season_n.reset_index(), on="season", how="left", validate="many_to_one"
)
season_summary["encounter_rate"] = (
    100 * season_summary["n_encounters"] / season_summary["total_censuses"]
)
assert season_summary["encounter_rate"].between(0, 100).all()
season_wide = season_summary.pivot(index="season", columns="Genus", values="encounter_rate")
season_wide = season_wide.reindex(index=season_order, columns=selected_genera)
display(season_summary)

## Shared bright colour palette

The charts use **two shades of blue for albatross genera** (*Diomedea*, *Thalassarche*), **two shades of red/coral for petrel genera** (*Pterodroma*, *Daption*), and **light green for the shearwater genus** (*Ardenna*). This conveys broad biological relationships while keeping a distinct legend entry for each genus. The related shades are intentionally brighter than the previous palette; value labels and thin outlines support readability. In the genus-first and small-panel alternatives, colours instead identify **seasons**, as stated in those legends.


## 4A. Version A — all genera in one grouped bar chart
Best for comparing genera within the same season and spotting overall seasonal differences. Use distinct colours, direct percentage labels, and sample sizes in the x-axis labels. This is the recommended static figure for your final HTML submission.

In [ ]:
import matplotlib.pyplot as plt

genus_palette = {'Diomedea': '#168BFF', 'Thalassarche': '#66C7FF', 'Ardenna': '#63D98E', 'Pterodroma': '#F04452', 'Daption': '#FF9370'}
colors = [genus_palette[g] for g in selected_genera]
fig, ax = plt.subplots(figsize=(12, 6.4))
season_wide.plot.bar(ax=ax, color=colors, width=0.78, edgecolor="#425466", linewidth=0.45)
ax.set_title("Seasonal encounter rates of five seabird genera", loc="left", fontsize=16, weight="bold", pad=20)
ax.set_ylabel("Full censuses recording genus (%)")
ax.set_xlabel("")
ax.set_xticklabels([f"{s.title()}\n(n={int(season_n[s]):,})" for s in season_order], rotation=0)
ax.set_ylim(0, min(100, max(10, float(season_wide.max().max()) * 1.18)))
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Genus", bbox_to_anchor=(1.01, 1), loc="upper left", frameon=False)
for bar in ax.patches:
    if bar.get_height() > 0:
        ax.annotate(f"{bar.get_height():.0f}%", (bar.get_x()+bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=7, xytext=(0, 3), textcoords="offset points")
fig.subplots_adjust(bottom=0.23, right=0.82)
fig.text(0.5, 0.015, "Note: n = all full 10-minute censuses in each season, including those without the selected genera.", ha="center", fontsize=9, color="dimgray")
plt.show()

## 4B. Version B — genus-focused interactive chart
This version lets the reader select one genus, while keeping all four seasons visible. It reduces colour clutter and makes changes across seasons easier to follow. It uses Plotly; export it separately as HTML if you want to share the interaction. A static screenshot or ordinary notebook HTML export may not preserve dropdown behaviour.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
for i, genus in enumerate(selected_genera):
    values = season_wide[genus].tolist()
    fig.add_trace(go.Bar(
        x=[f"{s.title()} (n={int(season_n[s]):,})" for s in season_order],
        y=values, name=genus, visible=(i == 0),
        marker_color=colors[i], text=[f"{v:.1f}%" for v in values],
        textposition="outside", hovertemplate="%{x}<extra>%{y:.1f}%</extra>"
    ))
buttons = [dict(label=g, method="update", args=[
    {"visible": [j == i for j in range(len(selected_genera))]},
    {"title": f"Seasonal encounter rate — {g}"}
]) for i, g in enumerate(selected_genera)]
fig.update_layout(
    title=f"Seasonal encounter rate — {selected_genera[0]}",
    yaxis_title="Full censuses recording genus (%)",
    yaxis=dict(range=[0, min(100, max(10, float(season_wide.max().max()) * 1.2))], gridcolor="#E8ECF0"),
    xaxis_title="Season", showlegend=False,
    updatemenus=[dict(buttons=buttons, direction="down", x=1.0, y=1.18, xanchor="right", yanchor="top")],
    margin=dict(t=105, r=25, b=115, l=65), template="plotly_white", height=480
)
fig.add_annotation(text="n = all full 10-minute censuses in each season (including censuses without these genera).", xref="paper", yref="paper", x=0.5, y=-0.30, showarrow=False, font=dict(size=11, color="gray"))
fig.show()
fig.write_html("seasonal_genus_dropdown.html", include_plotlyjs=True)
print("Saved interactive chart: seasonal_genus_dropdown.html")

# Alternative seasonal views — interpreting the teammate's suggestion

The comment was: *“have yall tried with groupby (Genus) instead of (Weather) and see the trends of birds across seasons?”*

**What our code actually does:** `groupby(["season", "Genus"])` already counts unique censuses for each genus within each season. We never group by `Weather` in this seasonal calculation.

**Possible interpretation 1 (tested below):** Reorganize the **chart** so genus is the main x-axis grouping, with four seasonal bars for each genus. This changes presentation only.

**Possible interpretation 2 (also tested):** Give each genus its own panel showing the four seasons, which makes within-genus patterns easier to see.

**Still worth clarifying at the meeting:** They may have meant a different *metric* or analysis, rather than changing the bar arrangement. We should not silently replace the agreed encounter-rate calculation based on an ambiguous comment.

Both alternatives reuse exactly the same `season_summary` and the same season-specific census denominators.

## Alternative C — group bars by genus

Each cluster now represents one genus, with a separate bar for spring, summer, autumn and winter. We still compare percentages (not bird totals), and each season uses its own census denominator. This view directly addresses the suggestion to focus on genus-level seasonal trends.
Here, colours distinguish seasons, rather than genus families; this is a different encoding from Versions A and B.


In [ ]:
# Pivot existing seasonal percentages; no cleaning or recalculation required.
genus_first = season_summary.pivot(index="Genus", columns="season", values="encounter_rate")
genus_first = genus_first.reindex(index=selected_genera, columns=season_order).fillna(0)

genus_first

In [ ]:
# Use one color per SEASON, rather than per genus, for this arrangement.
season_colors = ["#168BFF", "#63D98E", "#F04452", "#FFB347"]
ax = genus_first.plot(kind="bar", figsize=(12, 6), width=0.78, color=season_colors)
ax.set_title("Seasonal encounter rates within each seabird genus", fontsize=15, weight="bold", pad=16)
ax.set_xlabel("Genus")
ax.set_ylabel("Full censuses recording genus (%)")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Season", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.set_ylim(0, max(5, genus_first.to_numpy().max() * 1.17))
ax.grid(axis="y", alpha=0.18)
ax.set_axisbelow(True)
for container in ax.containers:
    ax.bar_label(container, fmt="%.1f%%", padding=2, fontsize=8)
fig = ax.get_figure()
fig.text(0.5, 0.015,
         "Encounter rate = censuses recording the genus / all full censuses in that season. "
         "n by season is printed below.",
         ha="center", fontsize=9, color="dimgray")
fig.subplots_adjust(bottom=0.22, right=0.82)
plt.show()

print("Full censuses per season (denominators):")
print(season_n)

## Alternative D — one small panel per genus

This layout removes the crowded 20-bar cluster. Each panel shows the four seasons for one genus, on a **shared vertical scale** so the heights remain comparable across panels. Use this when the main goal is to show how an individual genus changes through the seasons. The same five genera and encounter-rate values are used.
The same bright season palette is reused in every small panel.


In [ ]:
import math

fig, axes = plt.subplots(1, len(selected_genera), figsize=(17, 4.7), sharey=True)
max_rate = max(5, genus_first.to_numpy().max() * 1.18)
for ax, genus in zip(axes, selected_genera):
    values = genus_first.loc[genus, season_order]
    bars = ax.bar([s.title() for s in season_order], values, color=season_colors, width=0.7)
    ax.set_title(genus, fontstyle="italic", fontsize=12, weight="bold")
    ax.set_ylim(0, max_rate)
    ax.tick_params(axis="x", labelrotation=40, labelsize=8)
    ax.grid(axis="y", alpha=0.18)
    ax.set_axisbelow(True)
    for bar, value in zip(bars, values):
        ax.annotate(f"{value:.1f}%", (bar.get_x() + bar.get_width()/2, bar.get_height()),
                    xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8)
axes[0].set_ylabel("Full censuses recording genus (%)")
fig.suptitle("Seasonal encounter patterns across the five most frequently recorded genera",
             fontsize=15, weight="bold", y=1.04)
fig.text(0.5, 0.005,
         "Note: Rates use all eligible full censuses within each season; low values do not prove absence.",
         ha="center", fontsize=9, color="dimgray")
fig.tight_layout(rect=[0, 0.08, 1, 0.95])
plt.show()

## Check the proposed finding-based title against the data

The teammate suggested highlighting low Daption encounter rates in summer/autumn and low Ardenna encounter rates in winter. We should check the exact rates before using this as a headline. 'MIA' is too strong: a small rate is not necessarily zero, and these are observations from sampled ship censuses, not a complete census of all seabirds at sea.

The next cell prints the relevant rates and generates a cautious, data-dependent title suggestion. Compare it against the full graph and discuss geographic sampling differences in the interpretation.

In [ ]:
# Review the precise rates that would support a headline.
focus = genus_first.loc[[g for g in ["Daption", "Ardenna"] if g in genus_first.index]]
display(focus.round(2))

if {"Daption", "Ardenna"}.issubset(genus_first.index):
    d = genus_first.loc["Daption"]
    a = genus_first.loc["Ardenna"]
    if d[["summer", "autumn"]].max() < d[["spring", "winter"]].min() and a["winter"] == a.min():
        headline = "Daption encounters are lower in summer/autumn; Ardenna is least encountered in winter"
    else:
        headline = "Encounter rates vary by season and genus"
else:
    headline = "Encounter rates vary by season and genus"
print("Suggested cautious headline:", headline)
print("Interpretation caveat: differences in survey geography and effort may contribute to patterns.")